# الدرس السادس: الإخراج المهيكل الصارم باستخدام Pydantic (Structured Output with Pydantic)

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعلم كيفية اجبار نماذج اللغة على ارجاع استجابات مهيكلة بصيغة كائنات Pydantic مطابقة لمخططات بيانات محددة بدقة عبر الدالة الحديثة `model.with_structured_output`.

## اهمية الإخراج المهيكل في التطبيقات الانتاجية
- تحويل النصوص غير المهيكلة الى بيانات صالحة للادخال المباشر في قواعد البيانات ومحركات التحليل.
- التخلص تماما من مشاكل تلف صيغ JSON (Hallucinated syntax / Invalid JSON).
- الاستفادة من ميزات التحقق التلقائي من صحة الحقول (Field validation & constraints) المتوفرة في Pydantic V2.

## الخطوة 1: تهيئة البيئة واستيراد كلاسات Pydantic
نقوم باستيراد `BaseModel` و `Field` وتهيئة النموذج اللغوي.

In [ ]:
import os
from typing import List, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0)

## الخطوة 2: تعريف مخطط البيانات (Schema) عبر Pydantic
نعرف هيكلا لاستخراج بيانات الكتب ومراجعاتها مع تحديد اوصاف دقيقة وقيود على القيم.

In [ ]:
class BookAnalysis(BaseModel):
    title: str = Field(description="The title of the reviewed book.")
    author: str = Field(description="The author of the book.")
    publication_year: Optional[int] = Field(default=None, description="Year of original publication.")
    sentiment: str = Field(description="Sentiment of the review: Positive, Neutral, or Negative.")
    rating_out_of_ten: float = Field(ge=0, le=10, description="Numerical score from 0.0 to 10.0.")
    key_themes: List[str] = Field(description="List of primary themes discussed in the book.")

print("Schema Defined Successfully:")
print(BookAnalysis.model_json_schema())

## الخطوة 3: ربط النموذج بالمخطط عبر `with_structured_output`
نقوم بربط المخطط بالنموذج؛ حيث تعيد هذه الدالة كائنا جديدا يتولى داخليا تمرير المخطط للمزود وتحويل الرد مباشرة الى كائن Pydantic.

In [ ]:
structured_llm = model.with_structured_output(BookAnalysis)

review_text = """
I recently completed reading 'Clean Code' written by Robert C. Martin, published back in 2008.
The insights on writing readable, testable, and maintainable software are indispensable.
While some examples are Java-centric and slightly aged, the foundational principles remain gold.
Overall, I would give it an enthusiastic 8.8 out of 10. Key topics include meaningful naming,
refactoring, single responsibility principle, and test-driven development.
"""

# استدعاء النموذج لاستخراج البيانات
result = structured_llm.invoke(review_text)

print("Parsed Object Type:", type(result))
print("Title:", result.title)
print("Author:", result.author)
print("Year:", result.publication_year)
print("Sentiment:", result.sentiment)
print("Rating:", result.rating_out_of_ten)
print("Key Themes:", result.key_themes)

## الخطوة 4: التحويل الى قاموس وبيانات JSON
بما ان النتيجة عبارة عن كائن Pydantic نظامي، يمكننا استخدام دوال Pydantic الاصلية مثل `model_dump()` او `model_dump_json()` بكل سهولة.

In [ ]:
json_output = result.model_dump_json(indent=2)
print("Exported JSON Data:")
print(json_output)